John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

Import Packages

In [32]:
library(tidyverse)
# install.packages("tidymodels")
# install.packages('rlang')
# install.packages("kknn")
library(tidymodels)

Load data

In [33]:
data <- read_csv("../data/matching_opportunities.csv")

Rows: 245 Columns: 20
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl  (19): D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_M6, UST, M2_last_UST, M3...
dttm  (1): DateTime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [34]:
# Set data types
data <- data %>% 
    mutate(across(contains("UST"), ~ as.factor(.x)), 
        does_it_match = as.factor(does_it_match),
        number_of_matches = as.integer(number_of_matches)) %>%
    select(-DateTime, -number_of_matches)

In [35]:
glimpse(data)

Rows: 245
Columns: 18
$ D_M1_M2                 <dbl> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA…
$ D_M1_M3                 <dbl> 46.47336, 53.50176, 50.01110, 46.47336, 54.168…
$ D_M1_M4                 <dbl> 67.67739, 67.67739, 67.67739, 67.67739, 67.677…
$ D_M1_M5                 <dbl> 62.95599, 62.95599, 62.95599, 62.95599, 62.955…
$ D_M1_M6                 <dbl> 53.63760, 53.63760, 53.63760, 53.63760, 53.637…
$ UST                     <fct> 10, 5, 8, 10, 13, 120, 10, 10, 29, 29, 10, 8, …
$ M2_last_UST             <fct> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA…
$ M3_last_UST             <fct> 166, 35, 35, 35, 167, 3, 166, 27, 1, 10, 9, 20…
$ M4_last_UST             <fct> 3, 10, 22, 16, 5, 8, 60, 62, 62, 10, 3, 20, 16…
$ M5_last_UST             <fct> 10, 10, 33, 10, 10, 33, 20, 33, 33, 153, 20, 2…
$ M6_last_UST             <fct> 98, 98, 98, 9, 13, 10, 13, 10, 12, 10, 12, 9, …
$ time_since_M2_last_song <dbl> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA…
$ time_since_M3_la

In [36]:
# Data preprocessing happens in a recipe
recipe <- recipe(does_it_match ~ ., data = data,
    step_scale(all_numeric()) %>% # scale all numeric predictors
    step_center(all_numeric())) %>%  # center all numeric predictors
    prep()

In [37]:
# Create model specification
k = 5
model_spec <- nearest_neighbor(weight_func = "rectangular", neighbors=k) %>%
    set_engine("kknn") %>%
    set_mode("classification")

model <- model_spec %>%
    fit(does_it_match ~ ., data=bake(recipe, data))
model

parsnip model object


Call:
kknn::train.kknn(formula = does_it_match ~ ., data = data, ks = min_rows(5,     data, 5), kernel = ~"rectangular")

Type of response variable: nominal
Minimal misclassification: 0.4439252
Best kernel: rectangular
Best k: 5

In [49]:
head(data)

D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,D_M1_M6,UST,M2_last_UST,M3_last_UST,M4_last_UST,M5_last_UST,M6_last_UST,time_since_M2_last_song,time_since_M3_last_song,time_since_M4_last_song,time_since_M5_last_song,time_since_M6_last_song,does_it_match,time_rel_sunrise
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<dbl>
NA,46.47336,67.67739,62.95599,53.63760,10,NA,166,3,10,98,NA,16.62789,5.937839,17.629885,260.8946,1,-1094.3959
NA,53.50176,67.67739,62.95599,53.63760,5,NA,35,10,10,98,NA,22.56229,4.812736,23.564276,266.8290,0,-1088.4615
NA,50.01110,67.67739,62.95599,53.63760,8,NA,35,22,33,98,NA,39.13803,2.061269,6.152526,305.7566,0,-1049.5339
NA,46.47336,67.67739,62.95599,53.63760,10,NA,35,16,10,9,NA,15.95502,8.752150,65.999395,490.5590,1,-864.7315
NA,54.16852,67.67739,62.95599,53.63760,13,NA,167,5,10,13,NA,46.91866,16.897471,49.065689,955.3155,1,-399.9750
NA,52.93973,NA,62.95599,91.08596,120,NA,3,8,33,10,NA,22.85571,NA,0.658225,NA,0,-2005.6074


In [52]:
slice_tail(data, n=1)

D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,D_M1_M6,UST,M2_last_UST,M3_last_UST,M4_last_UST,M5_last_UST,M6_last_UST,time_since_M2_last_song,time_since_M3_last_song,time_since_M4_last_song,time_since_M5_last_song,time_since_M6_last_song,does_it_match,time_rel_sunrise
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<dbl>
78.37111,39.73681,67.67739,62.95599,53.6376,9,3,8,8,153,10,6.175548,8.935323,579.3218,-0.3307658,604.7416,0,-583.7896


In [ ]:
mutate(across(contains("UST"), ~ as.factor(.x)))

In [ ]:
# Test classifier on a random row in the DataFrame
test_observation <- slice_sample(data %>% mutate(across(where(is.numeric), ~ replace_na(.x, 0))), n=1)
print(test_observation)
test_prediction <- model %>% predict(test_observation)
print(test_prediction)

# A tibble: 1 × 18
  D_M1_M2 D_M1_M3 D_M1_M4 D_M1_M5 D_M1_M6 UST   M2_last_UST M3_last_UST
    <dbl>   <dbl>   <dbl>   <dbl>   <dbl> <fct> <fct>       <fct>      
1    78.4    39.7    61.2    63.0    53.6 9     18          9          
# ℹ 10 more variables: M4_last_UST <fct>, M5_last_UST <fct>, M6_last_UST <fct>,
#   time_since_M2_last_song <dbl>, time_since_M3_last_song <dbl>,
#   time_since_M4_last_song <dbl>, time_since_M5_last_song <dbl>,
#   time_since_M6_last_song <dbl>, does_it_match <fct>, time_rel_sunrise <dbl>
# A tibble: 1 × 1
  .pred_class
  <fct>      
1 1          
